In [3]:
import pandas as pd
import numpy as np

df = pd.read_csv("../data_processed/master_test_dataset.csv")

In [4]:
print(df.columns.tolist())

['match_id', 'team1', 'team2', 'venue', 'city', 'date', 'winner', 'win_by_runs', 'win_by_wickets', 'year', 'result_type', 'host_country', 'home_team', 'away_team']


In [5]:
df["date"] = pd.to_datetime(df["date"])
df["year"] = df["date"].dt.year

df.head()

print(df.columns.tolist())

['match_id', 'team1', 'team2', 'venue', 'city', 'date', 'winner', 'win_by_runs', 'win_by_wickets', 'year', 'result_type', 'host_country', 'home_team', 'away_team']


In [6]:
df_ncv = df[df['city'].isna()]
df_ncv['venue'].unique()

array([], dtype=object)

In [7]:
venue_city_map = {
    "Galle International Stadium": "Galle",
    "Adelaide Oval": "Adelaide",
    "Sydney Cricket Ground": "Sydney",
    "Melbourne Cricket Ground": "Melbourne",
    "Dubai International Cricket Stadium": "Dubai",
    "Harare Sports Club": "Harare",
    "Pallekele International Cricket Stadium": "Pallekele",
    "Sharjah Cricket Stadium": "Sharjah",
    "Chittagong Divisional Stadium": "Chittagong",
    "Perth Stadium": "Perth",
    "Sylhet International Cricket Stadium": "Sylhet",
    "Multan Cricket Stadium": "Multan",
}

df["city"] = df.apply(
    lambda row: venue_city_map.get(row["venue"], row["city"])
    if pd.isna(row["city"])
    else row["city"],
    axis=1
)

city_country_map = {

    # India
    "Mumbai": "India",
    "Delhi": "India",
    "Kanpur": "India",
    "Kolkata": "India",
    "Indore": "India",
    "Rajkot": "India",
    "Visakhapatnam": "India",
    "Chandigarh": "India",
    "Chennai": "India",
    "Hyderabad": "India",
    "Pune": "India",
    "Bengaluru": "India",
    "Bangalore": "India",
    "Ranchi": "India",
    "Dharamsala": "India",
    "Nagpur": "India",
    "Ahmedabad": "India",
    "Guwahati": "India",

    # Australia
    "Perth": "Australia",
    "Hobart": "Australia",
    "Brisbane": "Australia",
    "Melbourne": "Australia",
    "Sydney": "Australia",
    "Canberra": "Australia",
    "Adelaide": "Australia",
    "Darwin": "Australia",

    # England
    "London": "England",
    "Manchester": "England",
    "Birmingham": "England",
    "Leeds": "England",
    "Nottingham": "England",
    "Southampton": "England",
    "Chester-le-Street": "England",
    "Cardiff": "England",

    # New Zealand
    "Wellington": "New Zealand",
    "Christchurch": "New Zealand",
    "Hamilton": "New Zealand",
    "Dunedin": "New Zealand",
    "Auckland": "New Zealand",
    "Mount Maunganui": "New Zealand",
    "Napier": "New Zealand",

    # South Africa
    "Cape Town": "South Africa",
    "Johannesburg": "South Africa",
    "Durban": "South Africa",
    "Centurion": "South Africa",
    "Bloemfontein": "South Africa",
    "Port Elizabeth": "South Africa",
    "Potchefstroom": "South Africa",
    "Gqeberha": "South Africa",

    # Sri Lanka
    "Colombo": "Sri Lanka",
    "Galle": "Sri Lanka",
    "Pallekele": "Sri Lanka",
    "Kandy": "Sri Lanka",

    # Bangladesh
    "Dhaka": "Bangladesh",
    "Chittagong": "Bangladesh",
    "Chattogram": "Bangladesh",
    "Sylhet": "Bangladesh",
    "Khulna": "Bangladesh",
    "Bogra": "Bangladesh",
    "Fatullah": "Bangladesh",
    "Mirpur": "Bangladesh",

    # Pakistan
    "Karachi": "Pakistan",
    "Lahore": "Pakistan",
    "Rawalpindi": "Pakistan",
    "Multan": "Pakistan",
    "Faisalabad": "Pakistan",

    # Zimbabwe
    "Harare": "Zimbabwe",
    "Bulawayo": "Zimbabwe",

    # West Indies
    "Antigua": "West Indies",
    "Jamaica": "West Indies",
    "St Lucia": "West Indies",
    "Trinidad": "West Indies",
    "Barbados": "West Indies",
    "Dominica": "West Indies",
    "North Sound": "West Indies",
    "Bridgetown": "West Indies",
    "Gros Islet": "West Indies",
    "Kingston": "West Indies",
    "Roseau": "West Indies",
    "Port of Spain": "West Indies",
    "Providence": "West Indies",
    "St George's": "West Indies",
    "St John's": "West Indies",
    "St Kitts": "West Indies",
    "St Vincent": "West Indies",
    "Grenada": "West Indies",
    "Guyana": "West Indies",

    # Ireland
    "Dublin": "Ireland",
    "Belfast": "Ireland",

    # UAE Neutral Venues
    "Dubai": "Neutral",
    "Sharjah": "Neutral",
    "Abu Dhabi": "Neutral",
}

df["host_country"] = df["city"].map(city_country_map)

In [8]:
df[df["host_country"].isna()][["city", "venue"]]

,city,venue


In [9]:
df.head()

,match_id,team1,team2,venue,city,date,winner,win_by_runs,win_by_wickets,year,result_type,host_country,home_team,away_team
0,1000851,Australia,South Africa,Western Australia Cricket Association Ground,Perth,2016-11-03,South Africa,177.0,NaN,2016,Win,Australia,Australia,South Africa
1,1000853,Australia,South Africa,Bellerive Oval,Hobart,2016-11-12,South Africa,80.0,NaN,2016,Win,Australia,Australia,South Africa
2,1000855,Australia,South Africa,Adelaide Oval,Adelaide,2016-11-24,Australia,NaN,7.0,2016,Win,Australia,Australia,South Africa
3,1000881,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Brisbane,2016-12-15,Australia,39.0,NaN,2016,Win,Australia,Australia,Pakistan
4,1000883,Australia,Pakistan,Melbourne Cricket Ground,Melbourne,2016-12-26,Australia,18.0,NaN,2016,Win,Australia,Australia,Pakistan


In [10]:
def get_home_team(row):
    if row["team1"] == row["host_country"]:
        return row["team1"]
    elif row["team2"] == row["host_country"]:
        return row["team2"]
    else:
        return "Neutral"

df["home_team"] = df.apply(get_home_team, axis=1)

In [11]:
def get_away_team(row):
    if row["home_team"] == row["team1"]:
        return row["team2"]
    elif row["home_team"] == row["team2"]:
        return row["team1"]
    else:
        return "Neutral"

df["away_team"] = df.apply(get_away_team, axis=1)

In [12]:
df[
    ["team1", "team2", "city",
     "host_country", "home_team", "away_team"]
].sample(15)

,team1,team2,city,host_country,home_team,away_team
550,Australia,West Indies,Brisbane,Australia,Australia,West Indies
634,New Zealand,South Africa,Wellington,New Zealand,New Zealand,South Africa
705,India,Australia,Chennai,India,India,Australia
526,West Indies,England,Trinidad,West Indies,West Indies,England
354,Australia,South Africa,London,England,Neutral,Neutral
427,England,India,Nagpur,India,India,England
796,West Indies,New Zealand,Jamaica,West Indies,West Indies,New Zealand
555,Australia,Pakistan,Hobart,Australia,Australia,Pakistan
309,India,England,Visakhapatnam,India,India,England
275,India,Bangladesh,Chattogram,Bangladesh,Bangladesh,India


In [13]:
df[df["home_team"] == "Neutral"][
    ["city", "venue"]
].drop_duplicates()

,city,venue
37,Dubai,Dubai International Cricket Stadium
38,Abu Dhabi,Sheikh Zayed Stadium
39,Sharjah,Sharjah Cricket Stadium
198,Southampton,"The Rose Bowl, Southampton"
284,London,"Kennington Oval, London"
354,London,"Lord's, London"
562,London,Lord's
563,Leeds,Headingley


In [14]:
df.head(2)

,match_id,team1,team2,venue,city,date,winner,win_by_runs,win_by_wickets,year,result_type,host_country,home_team,away_team
0,1000851,Australia,South Africa,Western Australia Cricket Association Ground,Perth,2016-11-03,South Africa,177.0,NaN,2016,Win,Australia,Australia,South Africa
1,1000853,Australia,South Africa,Bellerive Oval,Hobart,2016-11-12,South Africa,80.0,NaN,2016,Win,Australia,Australia,South Africa


In [15]:
df[df['home_team'] == 'Neutral']

,match_id,team1,team2,venue,city,date,winner,win_by_runs,win_by_wickets,year,result_type,host_country,home_team,away_team
37,1050229,Pakistan,West Indies,Dubai International Cricket Stadium,Dubai,2016-10-13,Pakistan,56.0,NaN,2016,Win,Neutral,Neutral,Neutral
38,1050231,Pakistan,West Indies,Sheikh Zayed Stadium,Abu Dhabi,2016-10-21,Pakistan,133.0,NaN,2016,Win,Neutral,Neutral,Neutral
39,1050233,Pakistan,West Indies,Sharjah Cricket Stadium,Sharjah,2016-10-30,West Indies,NaN,5.0,2016,Win,Neutral,Neutral,Neutral
79,1120284,Sri Lanka,Pakistan,Sheikh Zayed Stadium,Abu Dhabi,2017-09-28,Sri Lanka,21.0,NaN,2017,Win,Neutral,Neutral,Neutral
80,1120285,Sri Lanka,Pakistan,Dubai International Cricket Stadium,Dubai,2017-10-06,Sri Lanka,68.0,NaN,2017,Win,Neutral,Neutral,Neutral
128,1157370,Pakistan,Australia,Dubai International Cricket Stadium,Dubai,2018-10-07,NaN,NaN,NaN,2018,Draw,Neutral,Neutral,Neutral
129,1157371,Pakistan,Australia,Sheikh Zayed Stadium,Abu Dhabi,2018-10-16,Pakistan,373.0,NaN,2018,Win,Neutral,Neutral,Neutral
130,1157381,New Zealand,Pakistan,Sheikh Zayed Stadium,Abu Dhabi,2018-11-16,New Zealand,4.0,NaN,2018,Win,Neutral,Neutral,Neutral
131,1157382,Pakistan,New Zealand,Dubai International Cricket Stadium,Dubai,2018-11-24,Pakistan,16.0,NaN,2018,Win,Neutral,Neutral,Neutral
132,1157383,New Zealand,Pakistan,Sheikh Zayed Stadium,Abu Dhabi,2018-12-03,New Zealand,123.0,NaN,2018,Win,Neutral,Neutral,Neutral


In [16]:
df.head()

,match_id,team1,team2,venue,city,date,winner,win_by_runs,win_by_wickets,year,result_type,host_country,home_team,away_team
0,1000851,Australia,South Africa,Western Australia Cricket Association Ground,Perth,2016-11-03,South Africa,177.0,NaN,2016,Win,Australia,Australia,South Africa
1,1000853,Australia,South Africa,Bellerive Oval,Hobart,2016-11-12,South Africa,80.0,NaN,2016,Win,Australia,Australia,South Africa
2,1000855,Australia,South Africa,Adelaide Oval,Adelaide,2016-11-24,Australia,NaN,7.0,2016,Win,Australia,Australia,South Africa
3,1000881,Australia,Pakistan,"Brisbane Cricket Ground, Woolloongabba",Brisbane,2016-12-15,Australia,39.0,NaN,2016,Win,Australia,Australia,Pakistan
4,1000883,Australia,Pakistan,Melbourne Cricket Ground,Melbourne,2016-12-26,Australia,18.0,NaN,2016,Win,Australia,Australia,Pakistan


In [17]:
country = "India"

In [18]:
df["opponent"] = df.apply(
    lambda row: row["team2"]
    if row["team1"] == country
    else row["team1"],
    axis=1
)

In [20]:
def get_location(row):

    if row["home_team"] == country:
        return "Home"

    elif row["away_team"] == country:
        return "Away"

    else:
        return "Neutral"

df["location"] = df.apply(get_location, axis=1)

In [21]:
def get_result(row):

    if pd.isna(row["winner"]):
        return "Draw"

    elif row["winner"] == country:
        return "Win"

    else:
        return "Loss"

df["result"] = df.apply(get_result, axis=1)

In [28]:
df['year'].unique()

array([2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026,
       2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2001,
       2002, 2003, 2004, 2015], dtype=int32)

In [ ]:
Countries ={
        "Australia",
        "Bangladesh",
        "England",
        "ICC World XI",
        "India",
        "Ireland",
        "New Zealand",
        "Pakistan",
        "South Africa",
        "Sri Lanka",
        "West Indies",
        "Zimbabwe"
    }



KeyError: 'home_team'

In [32]:
df.to_csv("../data_processed/master_test_dataset_1.csv", index=False)